# Dataset quality, with citations normalised by age


**What this is.** A quality profile for every dataset in the omics catalogue: the technical quality of its samples, how well its replicates agree, how completely our own pipeline processed it, and how much attention its paper received -- **citations per year since publication** (Einar, instruction 76).

**How to read it.** The components are shown side by side; the combined score is their average rank and is never shown without them. A well-cited dataset with poor replicate agreement should look different from a quiet one that reproduces cleanly.

**What citations are not.** Attention predicted host-facing at AUC 0.902 in this project without any biology, so the citation rate is a weight on a *dataset*, in the `attention` category, and never a feature of a *gene*.

Built 2026-11-06. Citation counts are fetched from Europe PMC now and stored with the retrieval date, because they change.

In [1]:
import json, os, time, urllib.parse, urllib.request
import numpy as np, pandas as pd
cat = pd.read_csv(os.path.join(REPO, 'parascape', 'data', 'omics_catalogue.tsv'), sep='\t')
cat['pubmed'] = cat['pubmed'].astype(str).str.extract(r'(\d{5,9})', expand=False)
print(len(cat), 'datasets;', cat.pubmed.notna().sum(), 'with a PubMed id;',
      cat.pubmed.nunique(), 'distinct papers')
cat[['source', 'accession', 'title', 'pubmed', 'date']].head()

692 datasets; 281 with a PubMed id; 195 distinct papers


,source,accession,title,pubmed,date
0,GEO,GSE360,Macrophages and dendritic cells exposed to phylogenetically distinct parasites,12663451,2003/03/26
1,GEO,GSE2596,Heart transplant biopsies,16794196,2005/10/24
2,GEO,GSE2946,"Synthesis vs. Abundance, Bradyzoite Development",15685165,2005/07/16
3,GEO,GSE2947,"Thiouracil Pulse-chase, Tachyzoites and Bradyzoites mRNA stability data",15685165,2005/07/16
4,GEO,GSE3117,Tachyzoite to bradyzoite differentiation,11994153,2005/08/13


## 1 · Citations, from Europe PMC

`citedByCount` and the first publication date for each PubMed id, fifty ids per query. The rate is citations divided by years since publication, with a floor of half a year so a paper from last month is not divided by nearly zero.

In [2]:
def europepmc(pmids):
    # Bracketed: without them the source filter binds only to the last id.
    q = '(' + ' OR '.join(f'EXT_ID:{p}' for p in pmids) + ') AND SRC:MED'
    url = ('https://www.ebi.ac.uk/europepmc/webservices/rest/search?'
           + urllib.parse.urlencode({'query': q, 'resultType': 'lite', 'format': 'json',
                                     'pageSize': 100}))
    for attempt in range(4):
        try:
            with urllib.request.urlopen(url, timeout=60) as r:
                return json.loads(r.read())['resultList']['result']
        except Exception:
            time.sleep(5 * (attempt + 1))
    return []
ids = sorted(cat.pubmed.dropna().unique())
rows = []; _dt_today = time.strftime('%Y-%m-%d')
for i in range(0, len(ids), 50):
    for r in europepmc(ids[i:i + 50]):
        rows.append(dict(pubmed=str(r.get('pmid')), cited_by=int(r.get('citedByCount') or 0),
                         first_published=r.get('firstPublicationDate') or '',
                         journal=r.get('journalTitle', ''), retrieved=_dt_today))
cites = pd.DataFrame(rows).drop_duplicates('pubmed')
today = pd.Timestamp(_dt_today)
pub = pd.to_datetime(cites.first_published, errors='coerce')
cites['years_out'] = ((today - pub).dt.days / 365.25).clip(lower=0.5)
cites['citations_per_year'] = cites.cited_by / cites.years_out
cites.to_csv(os.path.join(OUT, 'citations.tsv'), sep='\t', index=False)
print(len(cites), 'of', len(ids), 'papers found; median', round(cites.citations_per_year.median(), 1),
      'citations a year')
cites.sort_values('citations_per_year', ascending=False).head(10)

194 of 195 papers found; median 4.4 citations a year


,pubmed,cited_by,first_published,journal,retrieved,years_out,citations_per_year
85,27594426,773,2016-09-02,Cell,2026-11-06,10.18,75.96
146,31955846,237,2020-01-16,Cell,2026-11-06,6.806,34.82
18,22195747,363,2011-12-01,Immunity,2026-11-06,14.93,24.31
23,21199955,363,2011-01-03,J Exp Med,2026-11-06,15.84,22.91
15,22981537,318,2012-09-13,Immunity,2026-11-06,14.15,22.48
189,38093015,65,2023-12-13,Nature,2026-11-06,2.899,22.42
14,23041064,300,2012-10-04,Immunity,2026-11-06,14.09,21.29
91,25974303,228,2015-05-01,Cell Host Microbe,2026-11-06,11.52,19.79
59,30728393,146,2019-02-06,Sci Rep,2026-11-06,7.748,18.84
162,40759751,23,2025-08-04,Nat Metab,2026-11-06,1.257,18.3


### The publication date, checked against NCBI

The rate divides by the age of the paper, so a wrong date is a wrong rate. The NCBI E-utilities publication date is fetched for the same ids; a paper whose two years differ is flagged, and the earlier date is used.

In [3]:
def ncbi_dates(pmids):
    url = ('https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?'
           + urllib.parse.urlencode({'db': 'pubmed', 'id': ','.join(pmids), 'retmode': 'json'}))
    for attempt in range(4):
        try:
            with urllib.request.urlopen(url, timeout=60) as r:
                res = json.loads(r.read())['result']
            return {k: v.get('sortpubdate', '')[:10].replace('/', '-') for k, v in res.items() if k != 'uids'}
        except Exception:
            time.sleep(5 * (attempt + 1))
    return {}
ncbi = {}
for i in range(0, len(ids), 150):
    ncbi.update(ncbi_dates(ids[i:i + 150])); time.sleep(0.5)
cites['ncbi_date'] = cites.pubmed.map(ncbi).fillna('')
epmc_year = pd.to_datetime(cites.first_published, errors='coerce').dt.year
ncbi_year = pd.to_datetime(cites.ncbi_date, errors='coerce').dt.year
cites['date_disagrees'] = epmc_year.notna() & ncbi_year.notna() & (epmc_year != ncbi_year)
earliest = pd.concat([pd.to_datetime(cites.first_published, errors='coerce'),
                      pd.to_datetime(cites.ncbi_date, errors='coerce')], axis=1).min(axis=1)
cites['years_out'] = ((today - earliest).dt.days / 365.25).clip(lower=0.5)
cites['citations_per_year'] = cites.cited_by / cites.years_out
cites.to_csv(os.path.join(OUT, 'citations.tsv'), sep='\t', index=False)
print(int(ncbi_year.notna().sum()), 'dates from NCBI;', int(cites.date_disagrees.sum()), 'disagree on the year')
cites[cites.date_disagrees][['pubmed', 'first_published', 'ncbi_date', 'cited_by']]

194 dates from NCBI; 8 disagree on the year


,pubmed,first_published,ncbi_date,cited_by
17,22144491,2011-12-05,2012-03-01,22
26,21149591,2010-12-13,2011-03-01,47
28,21098240,2010-11-22,2011-02-01,25
29,21166903,2010-11-09,2011-01-01,66
42,17088349,2006-11-06,2007-02-01,24
173,39716984,2024-12-24,2025-02-01,8
188,38147552,2023-12-26,2024-01-02,32
189,38093015,2023-12-13,2024-01-01,65


## 2 · The measured components, per study

* **Sample quality**: the mean of the newest per-sample technical score *without* its replicate term (`quality_no_replicate`), so it does not restate the next column.

* **Replicate agreement**: the median Spearman of each contrast with itself across replicate splits (`expression_traits`), and the median correlation of each run with the other runs of its group (`replicate_rho`).

* **Reproduced elsewhere**: of the study contrasts that share a theme with another study, the share that some other study reproduces (Spearman at least 0.3). It needs a partner study, so it is missing more often than not.

* **Processing completeness**: runs our pipeline quantified out of the runs planned.

In [4]:
sq_dir = sorted(d for d in os.listdir(os.path.join(REPO, 'results')) if d.startswith('rna_sample_quality_'))[-1]
sq = pd.read_csv(os.path.join(REPO, 'results', sq_dir, 'sample_quality.tsv'), sep='\t')
sample_quality = sq.groupby('study').quality_no_replicate.mean().rename('sample_quality')
rep_dir = sorted(d for d in os.listdir(os.path.join(REPO, 'results')) if d.startswith('expression_traits_20')
                 and os.path.exists(os.path.join(REPO, 'results', d, 'replicate_agreement.tsv')))[-1]
rep = pd.read_csv(os.path.join(REPO, 'results', rep_dir, 'replicate_agreement.tsv'), sep='\t')
replicate = rep.groupby('study').spearman.median().rename('replicate_agreement')
# The second, wider measure: how well each run matches the other runs of its condition group.
replicate_rho = sq.groupby('study').replicate_rho.median().rename('replicate_rho')
cross = pd.read_csv(os.path.join(REPO, 'results', rep_dir, 'cross_study_agreement.tsv'), sep='\t')
cross = cross[cross.study_a != cross.study_b]
both = pd.concat([cross.rename(columns={'column_a': 'column', 'study_a': 'study'})[['study', 'column', 'spearman']],
                  cross.rename(columns={'column_b': 'column', 'study_b': 'study'})[['study', 'column', 'spearman']]])
best = both.groupby(['study', 'column']).spearman.max()
reproduced = (best >= 0.3).groupby(level='study').mean().rename('reproduced_elsewhere')
NAS = '/nas_mnt/carruthers/Einar/parascape_omics'
plan_path = os.path.expanduser('~/.cache/parascape-downloads/automation/priority-plan.jsonl')
if not os.path.exists(plan_path):
    plan_path = os.path.join(NAS, 'catalogue', 'plan.jsonl')
plan = pd.DataFrame([json.loads(l) for l in open(plan_path) if l.strip()])
plan['done'] = [os.path.exists(os.path.join(NAS, 'rnaseq', s, a, 'done.json'))
                for s, a in zip(plan.study, plan.run_accession)]
completeness = plan.groupby('study').done.mean().rename('processing_completeness')
print('sample quality from', sq_dir, '; replicate agreement from', rep_dir)
pd.concat([sample_quality, replicate, replicate_rho, reproduced, completeness], axis=1).describe().round(3)

sample quality from rna_sample_quality_2026_11_06 ; replicate agreement from expression_traits_2026_11_06


,sample_quality,replicate_agreement,replicate_rho,reproduced_elsewhere,processing_completeness
count,65,7,41,34,153
mean,0.806,0.629,0.969,0.611,0.429
std,0.273,0.449,0.078,0.38,0.495
min,0.028,-0.378,0.51,0,0
25%,0.767,0.715,0.973,0.344,0
50%,0.937,0.769,0.99,0.667,0
75%,0.968,0.847,0.997,1,1
max,0.995,0.885,0.999,1,1


## 3 · The profile, and the combined score

Each component is turned into its rank among the datasets that have it (0-1), and the combined score is the mean of the ranks available. Citations enter as the log of the rate.

In [5]:
prof = cat[['source', 'accession', 'title', 'pubmed', 'date']].copy()
prof = prof.merge(cites[['pubmed', 'cited_by', 'citations_per_year', 'first_published', 'retrieved']],
                  on='pubmed', how='left')
prof = prof.join(sample_quality, on='accession').join(replicate, on='accession')
prof = prof.join(replicate_rho, on='accession').join(reproduced, on='accession')
prof = prof.join(completeness, on='accession')
comp = {'sample_quality': prof.sample_quality, 'replicate_agreement': prof.replicate_agreement,
        'replicate_rho': prof.replicate_rho, 'reproduced_elsewhere': prof.reproduced_elsewhere,
        'processing_completeness': prof.processing_completeness,
        'citations': np.log1p(prof.citations_per_year)}
ranks = pd.DataFrame({k: v.rank(pct=True) for k, v in comp.items()})
prof['quality_score'] = ranks.mean(axis=1, skipna=True)
prof['components_available'] = ranks.notna().sum(axis=1)
prof['score_without_replicate'] = ranks.drop(columns=['replicate_agreement', 'replicate_rho']).mean(axis=1, skipna=True)
prof.to_csv(os.path.join(OUT, 'dataset_quality.tsv'), sep='\t', index=False)
prof.components_available.value_counts().sort_index()

components_available
0    316
1    287
2     30
3     23
4     19
5     15
6      2

## 4 · The gate: does the score predict replicate agreement?

Two measures of replicate agreement are the targets: the split-half agreement of each contrast with itself (few studies, the stricter test) and the median correlation of each run with the other runs of its group (`replicate_rho`, every study with replicates). The score built **without** either is set against each (Spearman across studies), and so is each component on its own.

**Held out by construction.** The score has no fitted weights -- it is an average of ranks -- so no dataset was used to fit it and every study is a held-out study. The per-sample score passed the same kind of check at 0.63; a dataset score that cannot is reported as a description, not a weight.

In [6]:
from scipy.stats import spearmanr
gate = []
for target in ('replicate_agreement', 'replicate_rho'):
    have = prof.dropna(subset=[target]).drop_duplicates('accession')
    for name in ('score_without_replicate', 'sample_quality', 'reproduced_elsewhere',
                 'processing_completeness', 'citations_per_year'):
        sub = have.dropna(subset=[name])
        varies = sub[name].nunique() > 1 and sub[target].nunique() > 1
        if len(sub) >= 5 and varies:
            rho, p = spearmanr(sub[name], sub[target])
            gate.append(dict(target=target, predictor=name, studies=len(sub),
                             spearman=round(float(rho), 3), p=round(float(p), 4)))
        else:
            gate.append(dict(target=target, predictor=name, studies=len(sub), spearman=np.nan,
                             p=np.nan))
gate = pd.DataFrame(gate)
gate.to_csv(os.path.join(OUT, 'gate.tsv'), sep='\t', index=False)
gate

,target,predictor,studies,spearman,p
0,replicate_agreement,score_without_replicate,7,0.75,0.0522
1,replicate_agreement,sample_quality,7,0.857,0.0137
2,replicate_agreement,reproduced_elsewhere,6,0.471,0.346
3,replicate_agreement,processing_completeness,7,NaN,NaN
4,replicate_agreement,citations_per_year,2,NaN,NaN
5,replicate_rho,score_without_replicate,41,0.315,0.0451
6,replicate_rho,sample_quality,41,0.472,0.0018
7,replicate_rho,reproduced_elsewhere,25,0.239,0.2501
8,replicate_rho,processing_completeness,41,-0.107,0.5059
9,replicate_rho,citations_per_year,16,-0.186,0.4904


## 5 · Reading it

* The citation rate is stored with its raw count and retrieval date; recompute it rather than trusting an old rate.

* A component that does not predict replicate agreement is still a description of the dataset, not a reason to weight it.

* Nothing here is a feature of any gene.